# Task 1.2 — Key Assumptions

**Paper:** "Object Detection with Discriminatively Trained Part-Based Models"  
**Authors:** Felzenszwalb, Girshick, McAllester, Ramanan (2010)  
**Venue:** IEEE TPAMI

---

This notebook identifies and analyzes **four key assumptions** that the DPM relies upon. For each assumption, we explain why the method requires it, provide a concrete scenario where the assumption is violated, and reference the relevant section of the paper.

---

## Assumption 1: Spatial Consistency of Parts (Star-Structured Topology)

**Assumption:**  
Object parts maintain a roughly consistent spatial arrangement relative to the object's root (center). Each part is connected only to the root in a **star graph** structure — parts do not interact with each other directly.

**Why the method requires it:**  
The DPM uses a star-structured model (Section 3) where each part's position is modeled independently given the root position. The deformation cost for part $i$ depends only on its displacement $(dx_i, dy_i)$ from the anchor position relative to the root — it does not depend on the positions of other parts. This star structure enables efficient inference via the **generalized distance transform**, which can optimize each part independently in linear time. If parts were allowed to interact (e.g., in a general graph), exact inference would become NP-hard and require approximate methods.

**Violation scenario:**  
Consider detecting a person performing a **cartwheel** or an extreme yoga pose. In these cases, the spatial arrangement of body parts (head, hands, feet) relative to the torso is drastically different from typical standing/walking poses. The star model cannot express constraints like "the left hand must be near the left foot" because parts only connect to the root, not to each other. This would lead to implausible part configurations and missed or false detections.

**Paper Reference:**  
- **Section 3** — defines the star-structured (single-component) model and states that part positions are independent given the root.
- **Section 3.2** — discusses mixture components as a partial remedy for viewpoint variation, but each component still uses a star structure.

---

## Assumption 2: HOG Features Provide Sufficient Discriminative Information

**Assumption:**  
The HOG feature representation captures enough information about local appearance to distinguish object parts from background and from parts of other categories. Specifically, the 31-dimensional HOG cell descriptor (capturing gradient orientations and contrast) is assumed to be a sufficiently rich representation.

**Why the method requires it:**  
The entire DPM pipeline — root filters, part filters, and deformation scoring — operates on HOG feature maps. The linear filters $F_0, F_1, \ldots, F_n$ compute dot products with HOG features. If HOG descriptors do not capture the relevant appearance variation (e.g., color, texture frequency, fine details), the filters cannot learn meaningful templates, and the deformation model cannot compensate for poor appearance features. The paper (Section 2.4) carefully designs an extended HOG variant with contrast-insensitive features and normalization, but it is still a hand-crafted, gradient-based representation.

**Violation scenario:**  
Objects distinguished primarily by **color** or **fine texture** rather than shape would be poorly served by HOG features. For example:
- Detecting ripe vs. unripe fruit (same shape, different color) — HOG discards color information.
- Distinguishing a zebra from a horse — the key difference is stripe texture at a frequency that HOG cells may not capture.
- Low-contrast objects against similar-gradient backgrounds (e.g., white cat on a white couch) would produce weak HOG responses.

**Paper Reference:**  
- **Section 2.4** — describes the HOG feature variant and its 31-dimensional cell descriptor.
- **Section 2.1** — feature representation as the foundation of the model.
- **Section 6, Table 5** — performance varies by category, with lower AP for categories less characterized by distinctive edges/shapes.

---

## Assumption 3: Small Deformation Assumption (Quadratic Penalty)

**Assumption:**  
Part displacements from their anchor positions are **small** relative to the root filter size. The deformation cost is modeled as a **quadratic function** of displacement, meaning the penalty grows rapidly with distance.

**Why the method requires it:**  
The deformation cost function $d_i \cdot (dx, dy, dx^2, dy^2)$ (Equation 3) is a quadratic spring model. Quadratic penalties are natural for modeling "parts stay close to expected positions with some flexibility." The squared terms $dx^2$ and $dy^2$ ensure that the cost increases rapidly for large displacements, effectively preventing parts from drifting too far. This is also what makes the distance transform optimization efficient — the quadratic form allows exact solutions in linear time.

However, this means the model **cannot handle large displacements** gracefully. If a part needs to move far from its anchor (e.g., an arm raised above the head vs. at the side), the quadratic penalty would suppress this configuration even if the part filter response is strong.

**Violation scenario:**  
Consider detecting **cats** in various poses — curled up sleeping vs. stretched out jumping. The spatial relationship between head, body, and legs changes dramatically across these poses. The quadratic deformation penalty would either:
- Be too tight: missing detections when parts are far from anchors.
- Be too loose: allowing false positive part placements on background.

The paper partially addresses this with mixture components (different root+part configurations for different poses), but each component still assumes small deformations.

**Paper Reference:**  
- **Section 3, Equation 3** — quadratic deformation cost function.
- **Section 2.3** — generalized distance transform relies on the quadratic form.
- **Figure 5** — learned deformation costs show elliptical (quadratic) patterns.

---

## Assumption 4: Training Bounding Box Annotations Are Reliable

**Assumption:**  
The bounding box annotations in the training data accurately localize objects and provide a reliable signal for training the root filter and initializing part positions.

**Why the method requires it:**  
During Latent SVM training (Section 4), positive examples are defined by the annotated bounding boxes. The root filter is initialized from these bounding boxes, and part positions (latent variables) are initialized based on the root placement. If bounding boxes are **noisy** (too loose, too tight, or offset), the initial root filter will be poor, the latent variable initialization will be wrong, and the coordinate-descent optimization may converge to a bad local optimum.

Furthermore, during the latent relabeling step, the algorithm selects the highest-scoring window that sufficiently overlaps with the annotated bounding box. If annotations are inaccurate, the model may learn from poorly localized examples.

**Violation scenario:**  
Crowd-sourced annotations (e.g., via Amazon Mechanical Turk) for challenging categories can be inconsistent:
- **Occluded objects**: Annotators may draw tight boxes around the visible part only, or estimate the full object extent differently.
- **Ambiguous boundaries**: For categories like "potted plant" or "dining table," bounding box extent is subjective.
- **Duplicate/missed annotations**: If some objects in a training image are not annotated, they will be treated as negatives during hard negative mining, teaching the model to suppress true objects.

**Paper Reference:**  
- **Section 4** — training requires bounding box labels; latent variables are initialized from them.
- **Section 4.1** — coordinate descent alternates between relabeling (depends on annotation quality) and parameter updates.
- **Section 5** — training on PASCAL VOC, which has human-annotated bounding boxes.

---

## Summary Table

| # | Assumption | Key Risk | Paper Section |
|---|-----------|----------|---------------|
| 1 | Star-structured part topology | Cannot model part-part interactions | Section 3 |
| 2 | HOG features are sufficient | Fails on color/texture-dependent objects | Section 2.4 |
| 3 | Small part deformations (quadratic) | Cannot handle extreme pose changes | Section 3, Eq. 3 |
| 4 | Reliable bounding box annotations | Poor training from noisy labels | Section 4 |